In [17]:
from datetime import date
from pathlib import Path
import os
import re

import pandas as pd

# ---- User settings ---------------------------------------------------------
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'code' else Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data'

# Set to None or 'all' to scan every data/merged* folder.
# Set to a specific folder to scan only that dataset, for example:
#   SEQKIT_INPUT_FOLDER = 'mergedseq5'
#   SEQKIT_INPUT_FOLDER = 'data/mergedseq5'
#   SEQKIT_INPUT_FOLDER = '/absolute/path/to/data/mergedseq5'
# The command-line wrapper sets SEQKIT_INPUT_FOLDER from MERGED_DIR.
# Interactive runs default to data/mergedseq5.
SEQKIT_INPUT_FOLDER = os.environ.get('SEQKIT_INPUT_FOLDER', 'data/mergedseq5')

# Raw seqkit locate files created by code/run_primer_umi_tracking.sh.
SEQKIT_HITS_GLOB = 'barcode*_primer_hits.tsv'

OUTPUT_DIR = DATA_ROOT / 'primertrack'


def resolve_search_root(folder):
    if folder is None or str(folder).strip().lower() in {'', 'all'}:
        return DATA_ROOT
    folder = Path(folder)
    if folder.is_absolute():
        return folder
    if folder.parts and folder.parts[0] == 'data':
        return PROJECT_ROOT / folder
    return DATA_ROOT / folder


SEQKIT_SEARCH_ROOT = resolve_search_root(SEQKIT_INPUT_FOLDER)


def output_dataset_label(search_root):
    search_root = Path(search_root)
    if search_root == DATA_ROOT:
        return 'all'
    if search_root.name.startswith('barcode'):
        return search_root.parent.name
    return search_root.name


RUN_DATE = date.today().strftime('%Y%m%d')
SEQKIT_OUTPUT_LABEL = output_dataset_label(SEQKIT_SEARCH_ROOT)
OUTPUT_BEST_HITS = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hits_{RUN_DATE}.txt'
OUTPUT_ALL_HITS = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_all_hits_with_metadata_{RUN_DATE}.txt'
OUTPUT_COUNTS_BY_BARCODE = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hit_counts_by_barcode_{RUN_DATE}.txt'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Seqkit search root: {SEQKIT_SEARCH_ROOT}')
print(f'Output label: {SEQKIT_OUTPUT_LABEL}')
print(f'Run date: {RUN_DATE}')
print(f'Output best-hit table: {OUTPUT_BEST_HITS}')

Project root: /mnt/c/Users/mftor/Documents/metabarcoding
Data root: /mnt/c/Users/mftor/Documents/metabarcoding/data
Seqkit search root: /mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5
Output label: mergedseq5
Run date: 20260630
Output best-hit table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt


In [18]:
# Discover raw seqkit locate outputs.
if not SEQKIT_SEARCH_ROOT.exists():
    raise FileNotFoundError(f'Seqkit search root does not exist: {SEQKIT_SEARCH_ROOT}')

if SEQKIT_SEARCH_ROOT == DATA_ROOT:
    seqkit_files = sorted(DATA_ROOT.glob(f'merged*/barcode*/{SEQKIT_HITS_GLOB}'))
elif SEQKIT_SEARCH_ROOT.name.startswith('barcode'):
    seqkit_files = sorted(SEQKIT_SEARCH_ROOT.glob(SEQKIT_HITS_GLOB))
else:
    seqkit_files = sorted(SEQKIT_SEARCH_ROOT.glob(f'barcode*/{SEQKIT_HITS_GLOB}'))

print(f'Seqkit output files found: {len(seqkit_files)}')
for path in seqkit_files[:10]:
    print(path)

if not seqkit_files:
    raise FileNotFoundError(f'No files matching {SEQKIT_HITS_GLOB!r} found under {SEQKIT_SEARCH_ROOT}')

Seqkit output files found: 24
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode01/barcode01_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode02/barcode02_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode03/barcode03_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode04/barcode04_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode05/barcode05_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode06/barcode06_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode07/barcode07_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode08/barcode08_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode09/barcode09_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode10/barcode10_primer_hits.tsv


In [19]:
IUPAC = {
    'A': set('A'),
    'C': set('C'),
    'G': set('G'),
    'T': set('T'),
    'U': set('T'),
    'R': set('AG'),
    'Y': set('CT'),
    'S': set('GC'),
    'W': set('AT'),
    'K': set('GT'),
    'M': set('AC'),
    'B': set('CGT'),
    'D': set('AGT'),
    'H': set('ACT'),
    'V': set('ACG'),
    'N': set('ACGT'),
}


def path_metadata(path):
    path = Path(path)
    parts = path.parts
    # returns the index for the path part that contains 'barcode' else returns None
    barcode_index = next((i for i, part in enumerate(parts) if part.startswith('barcode')), None)
    barcode_folder = parts[barcode_index] if barcode_index is not None else None
    merged_folder = parts[barcode_index - 1] if barcode_index is not None and barcode_index > 0 else None
    return merged_folder, barcode_folder


def normalize_seqkit_columns(df):
    # seqkit locate commonly writes: seqID, patternName, pattern, strand, start, end, matched.
    # Keep any additional columns if present.
    df.columns = [str(col).strip() for col in df.columns]
    rename = {}
    for col in df.columns:
        low = col.lower()
        if low in {'seqid', 'seq_id', 'read_id', 'name'}:
            rename[col] = 'seqID'
        elif low in {'patternname', 'pattern_name'}:
            rename[col] = 'patternName'
    return df.rename(columns=rename)


def iupac_mismatches(pattern, matched):
    if pd.isna(pattern) or pd.isna(matched):
        return pd.NA
    pattern = str(pattern).upper().replace('U', 'T')
    matched = str(matched).upper().replace('U', 'T')
    # length of the shortest sequence between pattern and matched, i.e. the matched
    length = min(len(pattern), len(matched))
    mismatches = abs(len(pattern) - len(matched))
    # sequence parts that could mathc or mismatch, to count matches ignoring ambiguous nucleotides
    matches = 0
    # estimate mismatches accounting for different lengths between pattern and match
    for p, m in zip(pattern[:length], matched[:length]):
        allowed = IUPAC.get(p, IUPAC['N'])
        # does not count N or ambiguous as a match
        if m in allowed:
            matches += 1
        else:
            mismatches += 1
    return mismatches

# calculates the non-ambiguous comparable nucleotides in pattern (primers are degenerated)
def informative_length(pattern):
    if pd.isna(pattern):
        return pd.NA
    pattern = str(pattern).upper().replace('U', 'T')
    informative = sum(1 for base in pattern if base != 'N')
    return informative or len(pattern)

# annotates file data to keep track of barcode, dataset, and experiment
def read_seqkit_file(path):
    df = pd.read_csv(path, sep='\t')
    df = normalize_seqkit_columns(df)
    merged_folder, barcode_folder = path_metadata(path)
    df['merged_folder'] = merged_folder
    df['barcode_folder'] = barcode_folder
    df['source_seqkit_tsv'] = str(path)
    return df

In [ ]:
# Process large SeqKit files in chunks; do not concatenate every hit in RAM.
CHUNK_SIZE = 200_000
# change to False if you want to avoid writing very large hits instead of 
# writing the best hits. Suitable for large seqkit locate outputs
# fro mlarge -max-mismatches values
WRITE_ALL_HITS = False #True
best_by_key = {}
raw_columns = []
total_rows = 0
unknown_direction_count = 0
all_hits_written = False

def best_score(record):
    mismatches = iupac_mismatches(record.get('pattern'), record.get('matched'))
    informative = informative_length(record.get('pattern'))
    matched_length = len(str(record.get('matched', '')))
    identity = None if pd.isna(informative) or not informative or mismatches is None else 1 - mismatches / informative
    try:
        start = float(record.get('start', ''))
    except (TypeError, ValueError):
        start = float('inf')
    record['mismatch_count'] = mismatches
    record['informative_pattern_length'] = informative
    record['matched_length'] = matched_length
    record['primer_identity'] = identity
    return (mismatches if mismatches is not None else float('inf'), -(identity if identity is not None else float('-inf')), -matched_length, start, str(record.get('patternName', '')))

for path in seqkit_files:
    merged_folder, barcode_folder = path_metadata(path)
    for chunk in pd.read_csv(path, sep='\t', chunksize=CHUNK_SIZE, dtype='string'):
        chunk = normalize_seqkit_columns(chunk)
        for column in chunk.columns:
            if column not in raw_columns:
                raw_columns.append(column)
        records = []
        for values in chunk.to_dict('records'):
            record = {key: ('' if pd.isna(value) else value) for key, value in values.items()}
            record.update({'merged_folder': merged_folder, 'barcode_folder': barcode_folder, 'source_seqkit_tsv': str(path)})
            direction_match = re.match(r'^(forward|reverse)(?:_|$)', str(record.get('patternName', '')))
            record['primer_direction'] = direction_match.group(1) if direction_match else ''
            if not record['primer_direction']:
                unknown_direction_count += 1
            else:
                key = tuple(record.get(name, '') for name in ['merged_folder', 'barcode_folder', 'seqID', 'primer_direction'])
                candidate_score = best_score(record)
                if key not in best_by_key or candidate_score < best_by_key[key][0]:
                    best_by_key[key] = (candidate_score, record.copy())
            records.append(record)
        total_rows += len(records)
        if WRITE_ALL_HITS:
            output_columns = raw_columns + [column for column in ['merged_folder', 'barcode_folder', 'source_seqkit_tsv', 'primer_direction', 'mismatch_count', 'informative_pattern_length', 'matched_length', 'primer_identity'] if column not in raw_columns]
            frame = pd.DataFrame(records).reindex(columns=output_columns)
            frame.to_csv(OUTPUT_ALL_HITS, sep='\t', index=False, mode='a' if all_hits_written else 'w', header=not all_hits_written)
            all_hits_written = True
    print(f'Processed {path}: {total_rows:,} rows so far')

print(f'Processed {len(seqkit_files)} files and {total_rows:,} raw hit rows')
print(f'Ignored {unknown_direction_count:,} hits without a forward/reverse patternName')
if WRITE_ALL_HITS:
    print(f'Wrote streamed all-hit table: {OUTPUT_ALL_HITS}')
else:
    print('Skipped all-hit output (WRITE_ALL_HITS=False)')

Raw seqkit hit rows: 13,106,750
Columns: ['seqID', 'patternName', 'pattern', 'strand', 'start', 'end', 'matched', 'merged_folder', 'barcode_folder', 'source_seqkit_tsv']


,seqID,patternName,pattern,strand,start,end,matched,merged_folder,barcode_folder,source_seqkit_tsv
0,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_90,ggtagtctgtatctctatgtgrktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
1,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_80,ggtagtatatatgtctatagarktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
2,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_76,ggtagtagagagcgtcgcgtgrktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
3,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,ggtagtatagagctctacatarktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
4,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,ggtagtatagagctctacatarktcaacmaatcataaagatattgg,-,251,296,ggtagtatagagctctacataattcaaccaatcataacagtatcgt,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...


In [21]:
# Scoring and best-hit selection were completed by the streaming parser.
# The next cell materializes best_by_key and writes today's compact table.
print(f'Directional best-hit records selected: {len(best_by_key):,}')
print(f'Best-hit table to write: {OUTPUT_BEST_HITS}')

Wrote all-hit table with metadata: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_all_hits_with_metadata_20260630.txt


,merged_folder,barcode_folder,seqID,patternName,mismatch_count,primer_identity,matched_length
0,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_90,7,0.847826,46
1,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_80,5,0.891304,46
2,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_76,7,0.847826,46
3,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,0,1.000000,46
4,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,5,0.891304,46


In [22]:
# Keep the best forward and reverse hit per read and write a wide table.
best_records = {}
for (_, _, _, direction), (_, record) in best_by_key.items():
    read_key = tuple(record.get(name, '') for name in ['merged_folder', 'barcode_folder', 'seqID'])
    best_records.setdefault(read_key, {})
    best_records[read_key].update({f'{direction}_{column}': record.get(column, '') for column in raw_columns + ['source_seqkit_tsv', 'mismatch_count', 'informative_pattern_length', 'matched_length', 'primer_identity'] if column not in ['merged_folder', 'barcode_folder', 'seqID']})
best_columns = ['merged_folder', 'barcode_folder', 'seqID'] + [f'{direction}_{column}' for direction in ['forward', 'reverse'] for column in raw_columns + ['source_seqkit_tsv', 'mismatch_count', 'informative_pattern_length', 'matched_length', 'primer_identity'] if column not in ['merged_folder', 'barcode_folder', 'seqID']]
best_rows = []
for read_key in sorted(best_records):
    row = dict(zip(['merged_folder', 'barcode_folder', 'seqID'], read_key))
    row.update(best_records[read_key])
    best_rows.append(row)
best_hits = pd.DataFrame(best_rows, columns=best_columns)
best_hits.to_csv(OUTPUT_BEST_HITS, sep='\t', index=False)
print(f'Reads with at least one directional best hit: {len(best_hits):,}')
print(f'Forward best hits: {best_hits["forward_patternName"].notna().sum():,}')
print(f'Reverse best hits: {best_hits["reverse_patternName"].notna().sum():,}')
print(f'Wrote best-hit table: {OUTPUT_BEST_HITS}')
best_hits.head()

Reads with at least one directional best hit: 3,727,055
Forward best hits: 2,845,997
Reverse best hits: 2,522,123
Wrote best-hit table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt


,seqID,forward_patternName,forward_pattern,forward_strand,forward_start,forward_end,forward_matched,merged_folder,barcode_folder,forward_source_seqkit_tsv,...,reverse_pattern,reverse_strand,reverse_start,reverse_end,reverse_matched,reverse_source_seqkit_tsv,reverse_mismatch_count,reverse_informative_pattern_length,reverse_matched_length,reverse_primer_identity
0,0000bc31-cdb1-4513-8018-c453ae7b47ba,forward_tagged_primer_24,ggtagcacacgcgcgctatatrktcaacmaatcataaagatattgg,+,71.0,116.0,ggtagcacacgcgcgctatatgttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000f46e-1613-40e5-93ca-2c312d61b67a,NaN,NaN,NaN,NaN,NaN,NaN,mergedseq5,barcode01,NaN,...,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,+,113.0,159.0,ggtagatatatgcacgctctataaacttctggatgtccaaaaaatca,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,0.0,47.0,47.0,1.0
2,0000f966-3c52-46f4-9573-bf8c8a5efcdf,forward_tagged_primer_55,ggtaggagactagagatagtgrktcaacmaatcataaagatattgg,-,111.0,156.0,ggtaggagactagagatagtggttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000122a7-5b12-4988-99a7-bdc6c540c2a3,forward_tagged_primer_46,ggtagctcacactctctcacarktcaacmaatcataaagatattgg,+,82.0,127.0,ggtagctcacactctctcacagttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016a97-d608-4421-887d-297dc3985c89,forward_tagged_primer_5,ggtagacacatatcgcactacrktcaacmaatcataaagatattgg,+,64.0,109.0,ggtagacacatctcgcactacagtcaaccaatcatgaaggatcttg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Summarise the selected best directional hits by primer and barcode.
summary_rows = [record for _, record in best_by_key.values()]
summary = (pd.DataFrame(summary_rows).groupby(['merged_folder', 'barcode_folder', 'patternName'], dropna=False).size().reset_index(name='read_count').sort_values(['merged_folder', 'barcode_folder', 'read_count'], ascending=[True, True, False]))
summary.to_csv(OUTPUT_COUNTS_BY_BARCODE, sep='\t', index=False)
print(f'Wrote summary table: {OUTPUT_COUNTS_BY_BARCODE}')

# Release large parser intermediates; best_hits and summary stay available.
del summary_rows, best_rows, best_records, best_by_key
summary.head(20)

Wrote summary table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hit_counts_by_barcode_20260630.txt


,merged_folder,barcode_folder,patternName,read_count
97,mergedseq5,barcode01,reverse_tagged_primer_2,278598
79,mergedseq5,barcode01,forward_tagged_primer_81,11532
12,mergedseq5,barcode01,forward_tagged_primer_20,10019
84,mergedseq5,barcode01,forward_tagged_primer_86,9872
75,mergedseq5,barcode01,forward_tagged_primer_78,8333
93,mergedseq5,barcode01,forward_tagged_primer_94,8066
87,mergedseq5,barcode01,forward_tagged_primer_89,7925
28,mergedseq5,barcode01,forward_tagged_primer_35,7875
82,mergedseq5,barcode01,forward_tagged_primer_84,7778
29,mergedseq5,barcode01,forward_tagged_primer_36,7528


In [24]:
# The generated tables remain in memory for exploration; no disk reload is needed.
best_hits.head()


Reloaded all_hits: 13,106,750 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_all_hits_with_metadata_20260630.txt
Reloaded best_hits: 3,727,055 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt
Reloaded summary: 2,356 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hit_counts_by_barcode_20260630.txt


,seqID,forward_patternName,forward_pattern,forward_strand,forward_start,forward_end,forward_matched,merged_folder,barcode_folder,forward_source_seqkit_tsv,...,reverse_pattern,reverse_strand,reverse_start,reverse_end,reverse_matched,reverse_source_seqkit_tsv,reverse_mismatch_count,reverse_informative_pattern_length,reverse_matched_length,reverse_primer_identity
0,0000bc31-cdb1-4513-8018-c453ae7b47ba,forward_tagged_primer_24,ggtagcacacgcgcgctatatrktcaacmaatcataaagatattgg,+,71.0,116.0,ggtagcacacgcgcgctatatgttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000f46e-1613-40e5-93ca-2c312d61b67a,NaN,NaN,NaN,NaN,NaN,NaN,mergedseq5,barcode01,NaN,...,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,+,113.0,159.0,ggtagatatatgcacgctctataaacttctggatgtccaaaaaatca,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,0.0,47.0,47.0,1.0
2,0000f966-3c52-46f4-9573-bf8c8a5efcdf,forward_tagged_primer_55,ggtaggagactagagatagtgrktcaacmaatcataaagatattgg,-,111.0,156.0,ggtaggagactagagatagtggttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000122a7-5b12-4988-99a7-bdc6c540c2a3,forward_tagged_primer_46,ggtagctcacactctctcacarktcaacmaatcataaagatattgg,+,82.0,127.0,ggtagctcacactctctcacagttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016a97-d608-4421-887d-297dc3985c89,forward_tagged_primer_5,ggtagacacatatcgcactacrktcaacmaatcataaagatattgg,+,64.0,109.0,ggtagacacatctcgcactacagtcaaccaatcatgaaggatcttg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
